In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# --- Passo 1: Carregar e selecionar ---
houses_file_path = "/kaggle/input/datasets/rishitaverma02/house-prices-advanced-regression-techniques/train (1).csv"
houses_data = pd.read_csv(houses_file_path)

features_names = ["GrLivArea", "OverallQual", "YearBuilt"]
colunas = features_names + ["SalePrice"]

# --- Passo 2: Conferir e dividir ---
print("Ausências por coluna:")
print(houses_data[colunas].isnull().sum())

total_inicial = len(houses_data)
houses_data = houses_data.dropna(subset=colunas)
print(f"Linhas incompletas removidas: {total_inicial - len(houses_data)}")

X = houses_data[features_names]
y = houses_data["SalePrice"]

# Divisão: 60% Treino, 40% Resto
train_X, resto_X, train_y, resto_y = train_test_split(
    X, y, test_size=0.40, random_state=42
)

# 50% de 40% = 20% do total para validação e 20% para treino
teste_X, val_X, teste_y, val_y = train_test_split(
    resto_X, resto_y, test_size=0.50, random_state=42
)

# --- Passo 3: Treinar os modelos isolados ---
houses_model_linear = LinearRegression()
houses_model_random = RandomForestRegressor(
    n_estimators=100, min_samples_leaf=3, random_state=42
)

houses_model_linear.fit(train_X, train_y)
houses_model_random.fit(train_X, train_y)

Ausências por coluna:
GrLivArea      0
OverallQual    0
YearBuilt      0
SalePrice      0
dtype: int64
Linhas incompletas removidas: 0


RandomForestRegressor(min_samples_leaf=3, random_state=42)

In [2]:
# --- Passo 4: Previsões e Combinação (Média com pesos iguais) ---
predict_linear = houses_model_linear.predict(teste_X)
predict_random = houses_model_random.predict(teste_X)
print("Linear Regression: ")
print(predict_linear)
print("----------------------------------------")
print("\n\nRandom Forest Regressor: ")
print(predict_random)
print("----------------------------------------")

Linear Regression: 
[160598.62565072 309285.27009641 283241.28559329 107024.06922434
 341996.1692925  286912.84488284 257850.29785766 158664.28311359
 224209.69888633 133927.05903    148293.4656553  132797.81889323
 165464.21121522 146131.29548574 197797.79009605 316780.49430839
 290575.52142207 211812.75170834 249120.74851355 322375.51289909
 152124.47349476 138562.70861746 126882.10369845 255063.6717983
 106687.04776628 136614.041712   209886.87783038 135126.52692294
 128121.87044858 162746.06603316 439795.28655502  82726.43783076
 126459.042908   208978.69092032 107131.8750287  129833.67255111
 307458.37450607  99536.28711602 120988.15497564  84105.93976988
 119036.76726118  84946.61973278 180713.80419184  91494.38473302
 292269.76506951 275323.00059087 232794.89093341 253098.68003886
 122090.70030021 214391.41530822 266216.1309315  212304.34600148
 120108.87040902 135007.63083702 244517.2353563  171669.66755959
 171156.15237429 181588.52188399 164344.31448115 149616.75797439
 23152

In [3]:
# Combinação direta dos dois vetores
houses_mean = (predict_linear + predict_random) / 2

# --- Passo 5: Comparar com a tabela de MAE (em dólares) ---
mae_linear = mean_absolute_error(val_y, predict_linear)
mae_random = mean_absolute_error(val_y, predict_random)
mae_combinado = mean_absolute_error(val_y, houses_mean)

tabela_comparativa = pd.DataFrame({
    "Modelo": ["Linear", "Floresta", "Média dos dois"],
    "MAE (USD)": [mae_linear, mae_random, mae_combinado]
})

display(tabela_comparativa)

,Modelo,MAE (USD)
0,Linear,77669.044912
1,Floresta,80592.072058
2,Média dos dois,78703.879080


In [4]:
#Avaliação final: retreine a opção escolhida com treino e validação juntos. Avalie uma vez no teste.
houses_test_path = "/kaggle/input/datasets/rishitaverma02/house-prices-advanced-regression-techniques/test (1).csv"
houses_test_data = pd.read_csv(houses_test_path)

features_names_test = ["GrLivArea", "OverallQual", "YearBuilt"]

test_X = houses_test_data[features_names_test]

#Para a combinação, retreine ambos e faça a média das previsões.
houses_model_linear.fit(resto_X, resto_y)
houses_model_random.fit(resto_X, resto_y)

predicts_linear = houses_model_linear.predict(test_X)
predicts_random = houses_model_random.predict(test_X)

houses_mean_test = (predicts_linear + predicts_random) / 2

mean_linear = sum(predicts_linear) /  len(predicts_linear)
mean_random = sum(predicts_random) / len(predicts_random)
mean_combinado = sum(houses_mean_test) / len(houses_mean_test)

tabela_comparativa_test = pd.DataFrame({
    "Modelo": ["Linear", "Floresta", "Valores"],
    "MAE (USD)": [mean_linear, mean_random, mean_combinado]
})

display(tabela_comparativa_test)


,Modelo,MAE (USD)
0,Linear,180155.545182
1,Floresta,179795.373347
2,Valores,179975.459265


### Como cada algoritmo funciona e como vocês uniram as previsões?
O algoritmo de regressão linear combina as colunas "GrLivArea", "OverallQual" e "YearBuilt" aplica pesos aprendidos a cada um para estimar um valor contínuo numérico (preço). Ela busca a relação linear que minimiza a soma dos erros quadráticos.
O algoritmo Random Forest é um conjunto de árvores de decisão, que cria 100 árvores diversificadas e estima o preço dos imóveis combinando as características (área, qualidade e idade
A união das previsões foi feita por média aritmética simples. "houses_mean = (predict_linear + predict_random) / 2"

### Qual modelo isolado teve menor MAE na validação? Informe os dois valores.
foi o de Regressão Linear: 77669.044912 USD

Random Forest: 80592.072058 USD

### A média melhorou, piorou ou empatou com o melhor modelo isolado? Informe a diferença em dólares.
A média das previsões piorou o resultado em comparação com o melhor modelo isolado, ela errou 1034.834168 USD a mais do que a Regressão Linear sozinha (78703.879080 - 77669.044912 = 1034.834168 USD)
### Qual opção foi escolhida e qual foi seu MAE no teste final?
Foi escolhido o algoritmo de regressão linear e o MAE no teste final: 180.155,545182.